# 🦖 Dino Run — PPO training on Google Colab

This notebook trains a **PPO** agent to play the *existing* Pygame Dino Run game.
It does not re-implement the game: the environment is a headless wrapper around the
same simulation the human-playable `main.py` runs.

**Runtime ▸ Change runtime type ▸ T4 GPU** before you start (CPU works too, just slower).

## Order of work (nothing here trains for hours before it is validated)
1. Get the bundle into Colab (upload `dist/dino_run_rl_colab.zip`)
2. Install dependencies
3. Headless probe — is pygame really running without a display?
4. **Benchmark** parallel environment counts → pick the number empirically
5. **Smoke test** — reset/step/observe/reward/actions + `check_env` + a real PPO update
6. Short training run (proves the policy actually learns)
7. Long training run (the expensive cell, at the end)
8. Evaluate against the untrained baselines, then download the model

> The CPU runs the pygame simulation (in worker processes), the GPU runs the small
> MLP policy/value network. Giving the GPU fewer/more environments does **not** make
> the physics faster — that is why step 4 measures instead of guessing.

## 0. Machine check

In [ ]:
import os, sys, platform, shutil

print('python      :', sys.version.split()[0], '|', platform.platform())
print('cpus        :', os.cpu_count(), '| ram GB:', round(shutil.disk_usage('/').total / 1e9, 1))

try:
    import torch
    print('torch       :', torch.__version__, '| cuda available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('cuda device :', torch.cuda.get_device_name(0))
except ImportError:
    print('torch       : not installed yet (comes with Colab)')

!nvidia-smi -L 2>/dev/null || echo 'no NVIDIA GPU visible'

## 1. Get the project into Colab

Locally run `python training/make_bundle.py --verify` to build `dist/dino_run_rl_colab.zip`,
then upload it here. The cell also looks in `/content` and in Google Drive before asking.

In [ ]:
import os, pathlib, shutil, zipfile

PROJECT_DIR = pathlib.Path('/content/dino_rl')
ZIP_NAME = 'dino_run_rl_colab.zip'
DRIVE_ZIP = pathlib.Path('/content/drive/MyDrive') / ZIP_NAME

def locate_zip():
    for candidate in (pathlib.Path('/content') / ZIP_NAME, DRIVE_ZIP):
        if candidate.is_file():
            return candidate
    return None

if (PROJECT_DIR / 'game').is_dir():
    print('project already extracted at', PROJECT_DIR)
else:
    archive = locate_zip()
    if archive is None:
        from google.colab import files  # noqa
        print('Upload', ZIP_NAME)
        uploaded = files.upload()
        archive = pathlib.Path('/content') / next(iter(uploaded))
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    PROJECT_DIR.mkdir(parents=True)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(PROJECT_DIR)
    print('extracted', archive, '->', PROJECT_DIR)

os.chdir(PROJECT_DIR)
print(sorted(p.name for p in PROJECT_DIR.iterdir()))

In [ ]:
# 2. Dependencies. On Colab torch is already a CUDA build -- keep it.
%cd /content/dino_rl
!python -m pip install -q -r requirements.txt
!python -c "import pygame, gymnasium, stable_baselines3, numpy, psutil, torch; print('pygame', pygame.version.ver, '| gymnasium', gymnasium.__version__, '| sb3', stable_baselines3.__version__, '| numpy', numpy.__version__, '| torch', torch.__version__)"

## 2. Headless probe

Proves that pygame initialises an off-screen display (SDL `dummy` driver), that no
window or audio device is needed, and that the environment can reset and step.

In [ ]:
import os
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['SDL_AUDIODRIVER'] = 'dummy'
os.environ['PYGAME_HIDE_SUPPORT_PROMPT'] = '1'

!python - <<'PY'
import numpy as np
from game.headless import describe_display
from rl.environment import DinoRunEnv
print('display     :', describe_display())
env = DinoRunEnv(frame_skip=4, seed=3)
obs, info = env.reset(seed=3)
print('obs shape   :', obs.shape, '| dtype', obs.dtype, '| in space', env.observation_space.contains(obs))
print('actions     :', env.action_meanings)
total = 0.0
for _ in range(300):
    obs, reward, terminated, truncated, info = env.step(1)
    total += reward
    if terminated or truncated:
        obs, info = env.reset()
print('300 steps   : reward', round(total, 2), '| score window', info['score'])
print('HEADLESS OK')
PY

## 3. Benchmark the number of parallel environments

Measures single-process vs multi-process backends over a range of environment counts,
collects CPU/RAM usage, and runs real PPO probes to measure *end-to-end samples/s*
(environment stepping alone is meaningless if PPO updates dominate).

The result is written to `models/benchmark_envs.json`, which is what `--envs auto`
consumes later.

In [ ]:
%cd /content/dino_rl
!python -u training/benchmark_envs.py --envs 1,2,4,8,16,32 --steps 12000 --max-seconds 15 --backends dummy,process --ppo-probe 2 --ppo-steps 8192 --device auto --json models/benchmark_envs.json

In [ ]:
import json, os
report = json.load(open('models/benchmark_envs.json'))
print(f"{'envs':>5} {'backend':>8} {'steps/s':>10} {'cpu %':>8} {'ram MB':>8}")
for row in report['results']:
    print(f"{row['n_envs']:>5} {row['backend']:>8} {row['steps_per_sec']:>10.0f} "
          f"{row['resources']['cpu_percent']['mean']:>8.1f} {row['resources']['rss_mb']['mean']:>8.1f}")
print()
for probe in report['ppo_probes']:
    print(f"PPO probe: {probe['n_envs']:>4} envs -> {probe['samples_per_sec']:>8.0f} samples/s "
          f"(update-bound {probe['update_seconds_estimate']:.1f}s of {probe['seconds']:.1f}s)")
print()
print('recommendation:', json.dumps(report['recommendation'], indent=2))
N_ENVS = int(report['recommendation']['n_envs'])
BACKEND = report['recommendation']['backend']
print('-> N_ENVS =', N_ENVS, '| backend =', BACKEND)
os.environ['N_ENVS'] = str(N_ENVS)

## 4. Smoke test — the gate before any real training

Checks headless mode, reset/step, observation validity, action meanings (jump / crouch /
fast-fall), reward anti-exploit properties, `check_env`, multi-env independence,
process-vs-dummy agreement, a real PPO update, policy change, checkpoint round-trip,
CUDA placement and the visual code path. **Do not continue if this fails.**

In [ ]:
%cd /content/dino_rl
!python -u training/smoke_test.py --timesteps 4096 --envs 4 --device auto

## 5. Short training run (proves the policy learns)

A few hundred thousand steps. The evaluation callback runs every 25k steps, so you will
see the mean episode reward climb. If it does not climb here, a longer run will not help.

In [ ]:
N_ENVS = globals().get('N_ENVS', 8)
BACKEND = globals().get('BACKEND', 'process')
print('training with', N_ENVS, 'environments via the', BACKEND, 'backend')

%cd /content/dino_rl
!python -u training/train_ppo.py --timesteps 200000 --envs {N_ENVS} --backend {BACKEND} --frame-skip 4 --n-steps 512 --batch-size 512 --n-epochs 6 --gamma 0.997 --ent-coef 0.005 --learning-rate 5e-4 --eval-every 25000 --checkpoint-every 100000 --device auto --seed 42 --output models/ppo_colab_short

In [ ]:
%cd /content/dino_rl
!python -u training/evaluate_ppo.py --model models/ppo_colab_short --episodes 10 --seed 1234 --frame-skip 4 --json models/ppo_colab_short/evaluation.json

## 6. Longer training run

Now the expensive cell. `TIMESTEPS` below is the total budget for the run; the run is
resumable, so if the Colab runtime dies, re-run this cell (it finds `latest.zip`
automatically via `--resume`).

Note the reward scale: with `--frame-skip 4` one agent step is 4 game frames, so keep
`n_steps * envs` large enough that each PPO update sees a few hundred steps per env.

In [ ]:
import pathlib

N_ENVS = globals().get('N_ENVS', 8)
BACKEND = globals().get('BACKEND', 'process')
TIMESTEPS = 3000000          # total budget for this run
OUTPUT = 'models/ppo_dino_colab'

# Optional: keep checkpoints on Google Drive so they survive a runtime restart.
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive  # noqa
    drive.mount('/content/drive')
    OUTPUT = '/content/drive/MyDrive/ppo_dino_colab'
    pathlib.Path(OUTPUT).mkdir(parents=True, exist_ok=True)

resume = f'--resume {OUTPUT}/latest.zip' if pathlib.Path(OUTPUT, 'latest.zip').is_file() else ''
print('output:', OUTPUT, '| resuming' if resume else '| fresh run', '| timesteps:', TIMESTEPS)

%cd /content/dino_rl
!python -u training/train_ppo.py --timesteps {TIMESTEPS} {resume} --envs {N_ENVS} --backend {BACKEND} --frame-skip 4 --n-steps 512 --batch-size 512 --n-epochs 6 --gamma 0.997 --ent-coef 0.005 --learning-rate 5e-4 --eval-every 250000 --eval-episodes 5 --checkpoint-every 500000 --device auto --seed 42 --output {OUTPUT}

## 7. Evaluate and download

The evaluation runs 20 episodes with fixed seeds and compares against three baselines:
`random`, `do_nothing`, and a scripted `geometric_heuristic`. Keep in mind that the
heuristic is a hand-written rule-based player - it is a *strong* baseline, and a short
PPO run may not beat it yet.

In [ ]:
%cd /content/dino_rl
!python -u training/evaluate_ppo.py --model {OUTPUT} --episodes 20 --seed 1234 --frame-skip 4 --json {OUTPUT}/evaluation.json

import json, pathlib, shutil
report = json.load(open(f'{OUTPUT}/evaluation.json'))
print(json.dumps(report['comparison'], indent=2))

# package the run (model + config + evaluation) for download
bundle = pathlib.Path(f'{OUTPUT}/dino_ppo_colab_results')
shutil.make_archive(str(bundle), 'zip', OUTPUT)
print('wrote', bundle.with_suffix('.zip'), bundle.with_suffix('.zip').stat().st_size / 1e6, 'MB')

from google.colab import files  # noqa
files.download(str(bundle.with_suffix('.zip')))

## Notes

* **Watch it play.** Colab has no display, so playback there is limited to
  `python training/play_agent.py --model models/ppo_dino_colab --selftest --frames 1200`
  (it drives the *visual* game off-screen). Download the model and run locally:
  `python training/play_agent.py --model models/ppo_dino_colab` to see the real window.
* **Resuming.** `--resume` continues from `latest.zip`; `best/best_model.zip` is the best
*evaluated* checkpoint and is what evaluation/playback use by default. Continuing a run
  can degrade the policy — the best-model file never regresses.
* **CPU only?** Everything works with `--device cpu`; the policy is a 14,624-parameter MLP,
  so the GPU matters much less than the number of CPU workers feeding the environment.
* **Reproducibility.** `--seed` fixes the obstacle sequence; every environment gets its own
  derived seed (env *i* → `seed * 1000003 + i`), and evaluation uses fixed seeds.